[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/4chan_catalog.ipynb)

# 4chan API: catalog

Read every live thread on a board with `catalog.json`. The catalog is the board's front pages as JSON: one entry per page, each holding the original post (OP) of every thread on that page and a preview of its last replies.

The 4chan API is read-only JSON. It needs no account and no key. The only
dependency is `requests`, which Google Colab has preinstalled.

Rules from the [API documentation](https://github.com/4chan/4chan-API):
at most one request per second, poll a thread no more often than every 10
seconds, and disclose 4chan as the source of anything you publish from it.
Boards can contain offensive and not-safe-for-work content. Read the
"Research considerations" section on the topic page before collecting.

In [1]:
import requests

In [2]:
api_url = "https://a.4cdn.org/{board}/catalog.json"
resp = requests.get(api_url.format(board="g"))
catalog = resp.json()
len(catalog)

11

In [3]:
page_1 = catalog[0]
page_1.keys()

dict_keys(['page', 'threads'])

In [4]:
len(page_1["threads"])

15

Each entry in `threads` is the OP of one thread plus a few summary fields. The first thread on page 1 of `/g/` is the board's sticky, which is why `sticky` and `closed` are set here.

In [5]:
thread = page_1["threads"][0]
summary_fields = ["no", "resto", "sticky", "closed", "time", "replies", "images", "last_modified"]
{field: thread.get(field) for field in summary_fields}

{'no': 105076684,
 'resto': 0,
 'sticky': 1,
 'closed': 1,
 'time': 1745612650,
 'replies': 3,
 'images': 3,
 'last_modified': 1745612922}

The fields you will use:

| Field | Meaning |
|---|---|
| `no` | Post number. For the OP this is also the thread ID |
| `resto` | The thread the post belongs to. 0 for an OP |
| `time` | Unix timestamp of the post |
| `now` | The same time as a string in US Eastern time. Use `time` |
| `sub`, `com` | Subject and body. `com` is HTML, not plain text |
| `filename`, `ext`, `tim` | The attached image. The file URL is `https://i.4cdn.org/{board}/{tim}{ext}` |
| `replies`, `images` | Counts for the whole thread |
| `last_replies` | At most the last 5 replies, not all of them |
| `last_modified` | Unix timestamp of the last change to the thread |

In [6]:
# last_replies is a preview: at most five replies, so the catalog alone
# never gives you a complete thread
busy = page_1["threads"][1]
busy["replies"], [(reply["no"], reply["time"]) for reply in busy["last_replies"]]

(159,
 [(109682328, 1788035889),
  (109682591, 1788038103),
  (109682621, 1788038448),
  (109682647, 1788038637),
  (109682659, 1788038686)])

In [7]:
# All live thread IDs on the board, across every page
thread_ids = [thread["no"] for page in catalog for thread in page["threads"]]
len(thread_ids), thread_ids[:5]

(151, [105076684, 109660156, 109646078, 109681974, 109680642])